In [18]:
import pennylane as qml
from pennylane import numpy as np


In [40]:
from sklearn.datasets import load_iris

data = load_iris()
X = data.data
y = data.target

print(X.shape)  # (150, 4)

(150, 4)


In [41]:
import random

X_min = X.min(axis=0)   # min of each column
X_max = X.max(axis=0)   # max of each column

X = (X - X_min) / (X_max - X_min)

X = X * (2*np.pi) - np.pi   # [-π, π]



y_state = []
for y_true in y:
    if y_true == 0:
        t = np.array([1/np.sqrt(2) + 0.00j, 1/np.sqrt(2) + 0.0j, 0.0 + 0.0j, 0.0 + 0.0j], requires_grad=True)
    elif y_true == 1:
        t = np.array([1/np.sqrt(2) + 0.0j, 0.0 + 0.0j, 1/np.sqrt(2) + 0.0j, 0.0 + 0.0j], requires_grad=True)
    else:
        t = np.array([1/np.sqrt(2) + 0.00j, 0.0 + 0.0j, 0.0 + 0.0j, 1/np.sqrt(2) + 0.0j], requires_grad=True)
    
    y_state.append(t)



# Example lists (columns)
A = X
B = y
C = y_state

# Combine into rows
rows = list(zip(A, B, C))

# Shuffle rows
random.shuffle(rows)

# Unzip back into columns
A_shuf, B_shuf, C_shuf = zip(*rows)

# (optional) convert back to lists
X= list(A_shuf)
y = list(B_shuf)
y_state = list(C_shuf)



X_train = X[:100]
y_train = y[:100]
y_state_train = y_state[:100]
X_test = X[100:]
y_test = y[100:]
y_state_test = y_state[100:]

In [78]:
import random

n = 80

np.random.seed(n)
random.seed(n)   # <-- this was missing

params = np.random.rand(12, requires_grad=True) #* (2*np.pi) - np.pi   # [-π, π]

In [79]:
dev = qml.device("default.qubit", wires = 2)

In [80]:
h = np.array([[0, 0, 0, 0],
              [0, 1, 0, 0], 
              [0, 0, 2, 0], 
              [0, 0, 0, 3]])

# Hermitian observable
H = qml.Hermitian(h, wires=[0, 1])

In [81]:
@qml.qnode(dev)
def circuit(params, x,):
    
    #qml.U3(x[0], x[1], x[2], 0)
    #qml.U3(x[1], x[2], x[3], 1)
    qml.RX(x[0], 0)
    qml.RX(x[1], 1)

    qml.RY(params[0], 0)
    qml.RY(params[1], 1)
    qml.CNOT([1, 0])
    qml.RZ(params[2], 0)
    qml.RZ(params[3], 1)

    qml.CNOT([0, 1])

    #qml.RY(params[4], 0)
    #qml.RY(params[5], 1)


    qml.RX(x[2], 0)
    qml.RX(x[3], 1)

    qml.RY(params[4], 0)
    qml.RY(params[5], 1)
    qml.CNOT([1, 0])
    qml.RZ(params[6], 0)
    qml.RZ(params[7], 1)

    qml.CNOT([0, 1])

    qml.RX(x[0] + x[2], 0)
    qml.RX(x[1] + x[3], 1)

    qml.RY(params[8], 0)
    qml.RY(params[9], 1)
    qml.CNOT([1, 0])
    qml.RZ(params[10], 0)
    qml.RZ(params[11], 1)

    qml.CNOT([0, 1])

    qml.RX(x[0] + x[3], 0)
    qml.RX(x[1] + x[2], 1)


        
    return qml.probs(wires = [0, 1])

In [25]:
def cost(params, X, y):
    loss = 0

    for x, y_true in zip(X, y):
        probs = circuit(params, x)

        # pick probability of correct class
        p = probs[y_true]

        # cross-entropy
        loss -= np.log(probs[y_true] + 1e-10)
        loss += 0.1 * probs[3]   # discourage unused state

    return loss / len(X)

In [26]:
def compute_accuracy(params, X_test, y_test):
    correct = 0

    for x, y_true in zip(X_test, y_test):
        probs = circuit(params, x)

        y_pred = np.argmax(probs[:3])   # only first 3 states

        if y_pred == y_true:
            correct += 1

    return correct / len(X_test)

In [83]:
opt = qml.AdamOptimizer(stepsize=0.05)

steps = 50

for i in range(steps):
    params = opt.step(lambda p: cost(p, X_train, y_train), params)

    if i % 2 == 0:
        train_loss = cost(params, X_train, y_train)
        train_acc = compute_accuracy(params, X_train, y_train)
        test_acc = compute_accuracy(params, X_test, y_test)

        print(f"Step {i:3d} | Loss: {train_loss:.6f} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

Step   0 | Loss: 1.754490 | Train Acc: 0.3400 | Test Acc: 0.2000
Step   2 | Loss: 1.550759 | Train Acc: 0.3500 | Test Acc: 0.2600
Step   4 | Loss: 1.392474 | Train Acc: 0.4500 | Test Acc: 0.2800
Step   6 | Loss: 1.254652 | Train Acc: 0.5200 | Test Acc: 0.4800
Step   8 | Loss: 1.133428 | Train Acc: 0.6700 | Test Acc: 0.6000
Step  10 | Loss: 1.002266 | Train Acc: 0.7500 | Test Acc: 0.7200
Step  12 | Loss: 0.906119 | Train Acc: 0.7800 | Test Acc: 0.7800
Step  14 | Loss: 0.846653 | Train Acc: 0.7800 | Test Acc: 0.7800
Step  16 | Loss: 0.818508 | Train Acc: 0.8100 | Test Acc: 0.7600
Step  18 | Loss: 0.807417 | Train Acc: 0.8100 | Test Acc: 0.8000
Step  20 | Loss: 0.799477 | Train Acc: 0.8300 | Test Acc: 0.8000
Step  22 | Loss: 0.787957 | Train Acc: 0.8100 | Test Acc: 0.8000
Step  24 | Loss: 0.772395 | Train Acc: 0.8100 | Test Acc: 0.8000
Step  26 | Loss: 0.754772 | Train Acc: 0.8100 | Test Acc: 0.8000
Step  28 | Loss: 0.737148 | Train Acc: 0.8200 | Test Acc: 0.8000
Step  30 | Loss: 0.720982

In [57]:
for i in range(30):
    probs = circuit(params, X[i])
    print(np.max(probs))

0.6275144468854017
0.8733925197808948
0.5080089830276919
0.814498448996034
0.7622290874563016
0.8706744574462414
0.7105512428103494
0.3107814451823794
0.5315167378868636
0.5756873768119941
0.5464310293105769
0.7573470606728183
0.6972968537792001
0.5029482237255759
0.6558302299628791
0.43582838852877337
0.4185806656821883
0.5582955799171725
0.8365988501726696
0.45836258980109285
0.4966603631378272
0.5553295293203034
0.539678412610359
0.8845473925639514
0.4308348469764839
0.6034523134067341
0.4914617381022709
0.42840770412051254
0.505349420238356
0.6975345825706664


In [58]:
print(params)

[ 0.37048003  1.13170367  0.16670447  0.06457289  1.3570999   1.73873549
  0.30024805 -0.25113245]


In [15]:
for x in X_train[:10]:
    print(circuit(params, x))

[0.41506486 0.08980715 0.03304884 0.46207915]
[0.20814277 0.40547933 0.29467342 0.09170448]
[0.04722857 0.4107128  0.26404905 0.27800959]
[0.04772518 0.46819907 0.00710674 0.476969  ]
[0.06306132 0.56129319 0.04399732 0.33164817]
[0.04094484 0.39195935 0.47961681 0.087479  ]
[0.061496   0.46578126 0.14241418 0.33030855]
[0.23467714 0.02804667 0.67732379 0.0599524 ]
[0.04498424 0.4073224  0.10536453 0.44232883]
[0.38118608 0.06447369 0.13810467 0.41623557]
